In [3]:

import pandas as pd

ratings = {'item': ['M1', 'M2', 'M3', 'M4', 'M1', 'M2', 'M3', 'M4', 'M1', 'M3', 'M2', 'M1', 'M4','M1','M2','M3'],
           'user': ['U1', 'U1', 'U1', 'U1', 'U2', 'U2', 'U2', 'U2', 'U3','U4', 'U4', 'U5', 'U5', 'U6', 'U6', 'U6'],
           'rating': [1,1,5,5,5,5,2,2,5,5,1,2,4,1,1,5]}

df = pd.DataFrame(ratings)
df

,item,user,rating
0,M1,U1,1
1,M2,U1,1
2,M3,U1,5
3,M4,U1,5
4,M1,U2,5
5,M2,U2,5
6,M3,U2,2
7,M4,U2,2
8,M1,U3,5
9,M3,U4,5


In [4]:
from surprise import Reader, Dataset

reader = Reader(rating_scale=(1, 5)) 
data = Dataset.load_from_df(df[['user', 'item', 'rating']], reader)

In [5]:
from surprise import KNNBasic

trainset = data.build_full_trainset()

sim_options = {
    'name': 'cosine', # similarity measure
    'user_based': True,  # this setting is for user-based CF
}

algo = KNNBasic(sim_options=sim_options)
algo.fit(trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


In [6]:
user_id = 'U4'
item_id = 'M4'

pred = algo.predict(user_id, item_id, verbose=True)

user: U4         item: M4         r_ui = None   est = 3.94   {'actual_k': 2, 'was_impossible': False}


In [7]:
testset = trainset.build_anti_testset()
predictions = algo.test(testset)

print(predictions)

[Prediction(uid='U3', iid='M2', r_ui=np.float64(3.125), est=np.float64(2.3333333333333335), details={'actual_k': 3, 'was_impossible': False}), Prediction(uid='U3', iid='M3', r_ui=np.float64(3.125), est=np.float64(4.0), details={'actual_k': 3, 'was_impossible': False}), Prediction(uid='U3', iid='M4', r_ui=np.float64(3.125), est=np.float64(3.6666666666666665), details={'actual_k': 3, 'was_impossible': False}), Prediction(uid='U4', iid='M1', r_ui=np.float64(3.125), est=np.float64(1.8581466328409297), details={'actual_k': 3, 'was_impossible': False}), Prediction(uid='U4', iid='M4', r_ui=np.float64(3.125), est=np.float64(3.9401555395139165), details={'actual_k': 2, 'was_impossible': False}), Prediction(uid='U5', iid='M2', r_ui=np.float64(3.125), est=np.float64(2.102303252963153), details={'actual_k': 3, 'was_impossible': False}), Prediction(uid='U5', iid='M3', r_ui=np.float64(3.125), est=np.float64(4.173272560277636), details={'actual_k': 3, 'was_impossible': False}), Prediction(uid='U6', i

In [8]:
from collections import defaultdict


def get_top_n(predictions, n=10):
    """Return the top-N recommendation for each user from a set of predictions.

    Args:
        predictions(list of Prediction objects): The list of predictions, as
            returned by the test method of an algorithm.
        n(int): The number of recommendation to output for each user. Default
            is 10.

    Returns:
    A dict where keys are user (raw) ids and values are lists of tuples:
        [(raw item id, rating estimation), ...] of size n.
    """

    # First map the predictions to each user.
    top_n = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        top_n[uid].append((iid, est))

    # Then sort the predictions for each user and retrieve the k highest ones.
    for uid, user_ratings in top_n.items():
        user_ratings.sort(key=lambda x: x[1], reverse=True)
        top_n[uid] = user_ratings[:n]

    return top_n


top_n = get_top_n(predictions, n=2)

# Print the recommended items for each user
for uid, user_ratings in top_n.items():
    print(uid, [iid for (iid, _) in user_ratings])

U3 ['M3', 'M4']
U4 ['M4', 'M1']
U5 ['M3', 'M2']
U6 ['M4']


In [9]:
from surprise.model_selection import cross_validate

cross_validate(algo, data, measures=["RMSE", "MAE"], cv=3, verbose=True)

Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Evaluating RMSE, MAE of algorithm KNNBasic on 3 split(s).

                  Fold 1  Fold 2  Fold 3  Mean    Std     
RMSE (testset)    2.9645  3.0697  2.9416  2.9919  0.0558  
MAE (testset)     2.8500  2.8909  2.6000  2.7803  0.1286  
Fit time          0.00    0.00    0.00    0.00    0.00    
Test time         0.00    0.00    0.00    0.00    0.00    


{'test_rmse': array([2.96451233, 3.06971342, 2.94157994]),
 'test_mae': array([2.85      , 2.89090909, 2.6       ]),
 'fit_time': (0.0008020401000976562,
  0.00010395050048828125,
  0.00011205673217773438),
 'test_time': (0.00026106834411621094,
  0.0001246929168701172,
  0.000213623046875)}